# Exploratory Data Analysis (EDA) - Tabular Regression

This notebook provides an in-depth analysis of the 53 anonymized features in .
Target Metric: **RMSE (Root Mean Squared Error)**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import root_mean_squared_error

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
%matplotlib inline

## 1. Data Integrity & Ingestion
Check dimensions, missing values (NaNs), and column data types.

In [ ]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("hidden_test.csv")
print(f"Train shape: {train_df.shape}")
print(f"Test shape:  {test_df.shape}")
print(f"Total NaNs in Train: {train_df.isnull().sum().sum()}")
print(f"Total NaNs in Test:  {test_df.isnull().sum().sum()}")
train_df.head()

## 2. Target Variable Distribution
Examine target distribution range and summary statistics.

In [ ]:
plt.figure(figsize=(10, 4))
sns.histplot(train_df["target"], bins=50, kde=True, color="teal")
plt.title("Target Variable Distribution")
plt.xlabel("target")
plt.ylabel("Count")
plt.show()
train_df["target"].describe()

## 3. Feature-Target Relationships & Discoveries
Analyze linear and non-linear dependencies between features and .

In [ ]:
sample = train_df.sample(n=3000, random_state=42)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(sample["6"], sample["target"], alpha=0.3, s=10, color="darkblue")
axes[0].set_title("Feature 6 vs Target (Clear Quadratic Curve)")
axes[0].set_xlabel("Feature 6")
axes[0].set_ylabel("Target")

axes[1].scatter(sample["6"]**2, sample["target"], alpha=0.3, s=10, color="crimson")
axes[1].set_title("Feature 6^2 vs Target (Perfect Linear Mapping)")
axes[1].set_xlabel("Feature 6 squared")
axes[1].set_ylabel("Target")
plt.tight_layout()
plt.show()

## 4. Residual Inspection & Ground Truth Recovery
Subtracting  reveals that the remaining residual is exactly .

In [ ]:
diff = train_df["target"] - (train_df["6"]**2 + train_df["7"])
print(f"Max absolute error from formula (6^2 + 7): {np.max(np.abs(diff)):.2e}")
print(f"Mean absolute error: {np.mean(np.abs(diff)):.2e}")

## 5. Model Benchmarking
Comparison between Standard Linear Regression, Gradient Boosted Trees, and the Feature-Engineered Model.

In [ ]:
X = train_df.drop(columns=["target"])
y = train_df["target"]
kf = KFold(n_splits=5, shuffle=True, random_state=42)

lr_baseline_rmse = -cross_val_score(LinearRegression(), X, y, cv=kf, scoring="neg_root_mean_squared_error").mean()
print(f"1. Raw Linear Regression RMSE:           {lr_baseline_rmse:.4f}")

sample_idx = np.random.RandomState(42).choice(len(X), 20000, replace=False)
gbt_rmse = -cross_val_score(HistGradientBoostingRegressor(random_state=42), X.iloc[sample_idx], y.iloc[sample_idx], cv=3, scoring="neg_root_mean_squared_error").mean()
print(f"2. Gradient Boosted Trees (GBDT) RMSE:   {gbt_rmse:.4f}")

X_eng = X.copy()
X_eng["6_sq"] = X_eng["6"] ** 2
final_rmse = -cross_val_score(Ridge(alpha=1e-5), X_eng, y, cv=kf, scoring="neg_root_mean_squared_error").mean()
print(f"3. Discovered Non-Linear Model RMSE:     {final_rmse:.2e}")